# Actividad 01 — Semana 03: SQL en Databricks sobre tablas Delta

**Semana:** 03  
**Tema:** SQL sobre tablas Delta — SELECT, filtros, agrupaciones, HAVING  
**Estudiante:** Daniel Guzmán  
**Notebook:** sql_basico_daniel  

## Objetivo

Consumir las tablas Delta construidas en Semana 02 usando SQL en Databricks.

En esta actividad se trabajará principalmente con:

- `workspace.silver.transactions_daniel`
- `workspace.gold.fraude_por_categoria_daniel`
- `workspace.gold.fraude_por_tarjeta_daniel`
- `workspace.gold.fraude_temporal_daniel`
- `workspace.gold.usuarios_riesgo_daniel`

Se usa el sufijo `_daniel` para evitar conflictos con tablas de otros estudiantes en el catálogo compartido.

In [0]:
USE CATALOG workspace;

SHOW TABLES IN silver;

In [0]:
SHOW TABLES IN gold;

In [0]:
DESCRIBE TABLE silver.transactions_daniel;

In [0]:
DESCRIBE TABLE EXTENDED silver.transactions_daniel;

In [0]:
SELECT COUNT(*) AS total_filas
FROM silver.transactions_daniel;

In [0]:
SELECT *
FROM silver.transactions_daniel
LIMIT 10;

## Parte 0 — Exploración inicial

Se validaron las tablas disponibles en los schemas `silver` y `gold`.

En `silver` está disponible la tabla:

- `silver.transactions_daniel`

En `gold` están disponibles tablas como:

- `gold.fraude_por_categoria_daniel`
- `gold.fraude_por_tarjeta_daniel`
- `gold.fraude_temporal_daniel`
- `gold.usuarios_riesgo_daniel`
- `gold.resumen_fraude_daniel`
- `gold.fraude_por_dimension_daniel`
- `gold.fraude_por_monto_daniel`
- `gold.perfil_usuario_daniel`

La tabla principal `silver.transactions_daniel` tiene **13,305,915 filas** y **48 columnas**.

Tipos de datos relevantes:

- `amount`: double
- `transaction_date`: timestamp
- `is_fraud`: int

Esta tabla será la base para las consultas SQL de análisis porque contiene las transacciones enriquecidas con información de usuarios, tarjetas, categorías MCC y etiquetas de fraude.

In [0]:
SELECT
    transaction_id,
    transaction_date,
    amount,
    merchant_id,
    merchant_category,
    card_type
FROM silver.transactions_daniel
WHERE is_fraud = 1
  AND amount > 500
ORDER BY amount DESC
LIMIT 20;

In [0]:
SELECT *
FROM silver.transactions_daniel
WHERE mes = 1
ORDER BY transaction_date
LIMIT 20;

In [0]:
SELECT
    transaction_id,
    merchant_id,
    merchant_category,
    amount,
    is_fraud
FROM silver.transactions_daniel
WHERE LOWER(merchant_category) LIKE '%food%'
   OR LOWER(merchant_category) LIKE '%restaurant%'
ORDER BY amount DESC
LIMIT 50;

In [0]:
SELECT COUNT(*) AS sin_categoria
FROM silver.transactions_daniel
WHERE merchant_category IS NULL;

In [0]:
SELECT COUNT(*) AS sin_label_fraude
FROM silver.transactions_daniel
WHERE is_fraud IS NULL;

## Parte 1 — SELECT, WHERE y ORDER BY

Se aplicaron filtros básicos sobre `silver.transactions_daniel` usando `WHERE`, ordenamientos con `ORDER BY` y límites con `LIMIT`.

También se consultaron valores nulos en columnas importantes como `merchant_category` e `is_fraud`.

Los registros sin etiqueta de fraude (`is_fraud IS NULL`) no deberían eliminarse automáticamente en un análisis real, porque siguen siendo transacciones válidas. Sin embargo, para calcular tasas de fraude, conviene excluirlos del denominador y usar únicamente transacciones etiquetadas (`is_fraud = 0` o `is_fraud = 1`) para evitar diluir artificialmente la tasa.

En la validación de nulos se encontró que no hay transacciones sin categoría de comercio (`sin_categoria = 0`), pero sí existen **4,390,952 transacciones sin etiqueta de fraude**. Por eso, para análisis de tasa de fraude es mejor usar solo registros etiquetados.

In [0]:
SELECT
    card_type,
    COUNT(*) AS total_transacciones,
    ROUND(SUM(amount), 2) AS monto_total,
    ROUND(AVG(amount), 2) AS ticket_promedio,
    ROUND(MAX(amount), 2) AS monto_maximo
FROM silver.transactions_daniel
GROUP BY card_type
ORDER BY total_transacciones DESC;

In [0]:
SELECT
    card_type,
    COUNT(*) AS total_transacciones,
    ROUND(SUM(amount), 2) AS monto_total,
    ROUND(AVG(amount), 2) AS ticket_promedio,
    ROUND(MAX(amount), 2) AS monto_maximo
FROM silver.transactions_daniel
GROUP BY card_type
ORDER BY total_transacciones DESC;

In [0]:
SELECT
    anio,
    mes,
    COUNT(*) AS total_transacciones,
    SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) AS total_fraudes,
    SUM(CASE WHEN is_fraud IN (0, 1) THEN 1 ELSE 0 END) AS transacciones_etiquetadas,
    ROUND(
        SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END)
        / SUM(CASE WHEN is_fraud IN (0, 1) THEN 1 ELSE 0 END) * 100,
        4
    ) AS tasa_fraude_pct
FROM silver.transactions_daniel
GROUP BY anio, mes
ORDER BY tasa_fraude_pct DESC;

In [0]:
SELECT
    merchant_category,
    COUNT(*) AS total_transacciones,
    SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) AS total_fraudes,
    SUM(CASE WHEN is_fraud IN (0, 1) THEN 1 ELSE 0 END) AS transacciones_etiquetadas,
    ROUND(
        SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END)
        / SUM(CASE WHEN is_fraud IN (0, 1) THEN 1 ELSE 0 END) * 100,
        4
    ) AS tasa_fraude_pct
FROM silver.transactions_daniel
GROUP BY merchant_category
HAVING COUNT(*) > 1000
   AND (
        SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END)
        / SUM(CASE WHEN is_fraud IN (0, 1) THEN 1 ELSE 0 END) * 100
   ) > 5
ORDER BY tasa_fraude_pct DESC;

## Resultados Parte 2

El pico de fraude mensual se observó en **diciembre de 2015**, con una tasa de fraude de **0.5278%**, **423 fraudes** y **80,147 transacciones etiquetadas**.

Al aplicar `HAVING` para buscar categorías con más de 1,000 transacciones y tasa de fraude mayor al 5%, la categoría con mayor tasa fue **Computers, Computer Peripheral Equipment**, con **10.8338%** de tasa de fraude, **204 fraudes** y **1,883 transacciones etiquetadas**.

Este análisis muestra cómo `HAVING` permite filtrar sobre métricas agregadas, algo que no se puede hacer directamente con `WHERE`.

## Parte 2 — GROUP BY y HAVING

Se usaron agregaciones con `GROUP BY` para resumir transacciones por tipo de tarjeta, mes y categoría de comercio.

La diferencia entre `WHERE` y `HAVING` es que:

- `WHERE` filtra filas antes de agrupar.
- `HAVING` filtra resultados después de agrupar.

Por eso `HAVING` se usa cuando el filtro depende de una agregación, por ejemplo `COUNT(*) > 1000` o una tasa calculada con `SUM(...) / COUNT(...)`.

En SQL normalmente no se puede usar directamente el alias `tasa_fraude_pct` dentro del mismo `HAVING` donde se define, porque el orden lógico de ejecución evalúa `HAVING` antes de exponer los alias del `SELECT`. Por eso se repite la expresión completa o se usa una subconsulta.

In [0]:
SELECT
    hora,
    COUNT(*) AS total_transacciones,
    SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) AS fraudes,
    SUM(CASE WHEN is_fraud IN (0, 1) THEN 1 ELSE 0 END) AS transacciones_etiquetadas,
    ROUND(
        SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END)
        / SUM(CASE WHEN is_fraud IN (0, 1) THEN 1 ELSE 0 END) * 100,
        4
    ) AS tasa_fraude_pct
FROM silver.transactions_daniel
GROUP BY hora
ORDER BY hora;

In [0]:
SELECT
    MIN(transaction_date) AS primera_transaccion,
    MAX(transaction_date) AS ultima_transaccion,
    DATEDIFF(MAX(transaction_date), MIN(transaction_date)) AS dias_de_historia
FROM silver.transactions_daniel;

In [0]:
SELECT *
FROM silver.transactions_daniel
WHERE transaction_date >= DATE_ADD(
    (SELECT MAX(transaction_date) FROM silver.transactions_daniel),
    -30
)
ORDER BY transaction_date DESC
LIMIT 30;

## Parte 3 — Funciones de fecha en SQL

Se usaron funciones temporales sobre `transaction_date` y columnas derivadas como `hora`.

Con `DATEDIFF` se calculó la cantidad de días entre la primera y última transacción registrada en el dataset.

Con `DATE_ADD` se filtraron las transacciones correspondientes a los últimos 30 días del periodo disponible en los datos, tomando como referencia la fecha máxima registrada y no la fecha actual del sistema.

El dataset contiene un historial de **3,590 días**, desde **2010-01-01 00:01:00** hasta **2019-10-31 23:59:00**.

In [0]:
SELECT *
FROM gold.fraude_por_tarjeta_daniel
ORDER BY tasa_fraude_pct DESC;

In [0]:
SELECT
    merchant_category,
    mcc,
    total_transacciones,
    total_fraudes,
    transacciones_etiquetadas,
    tasa_fraude_pct
FROM gold.fraude_por_categoria_daniel
ORDER BY tasa_fraude_pct DESC
LIMIT 10;

## Parte 4 — Consultas sobre tablas Gold

Las tablas `gold` son útiles cuando se necesitan respuestas rápidas sobre métricas ya agregadas, por ejemplo fraude por tipo de tarjeta, categoría de comercio o periodo temporal.

Conviene consultar `gold.*` cuando:

- La pregunta de negocio ya está cubierta por una tabla agregada.
- Se necesita consultar rápido sin recalcular sobre millones de filas.
- Se quiere entregar una vista limpia a usuarios analíticos.

Conviene consultar `silver.transactions_daniel` cuando:

- Se necesita mayor nivel de detalle.
- Se requiere construir una agregación nueva.
- Se necesita validar o auditar cómo se calculó una métrica.
- La tabla Gold no contiene las columnas necesarias para responder la pregunta.

Usar Gold directamente puede ser mala idea si no se conoce cómo fue construida la agregación, qué filtros aplicó o qué registros quedaron excluidos, especialmente en este dataset donde existen transacciones sin etiqueta de fraude.

### Resultado Gold — tipo de tarjeta

Al consultar `gold.fraude_por_tarjeta_daniel`, el tipo de tarjeta con mayor tasa de fraude fue **Debit (Prepaid)**, con **1,378 fraudes** sobre **613,300 transacciones etiquetadas**.

In [0]:
SELECT
    CASE
        WHEN amount_abs < 10    THEN 'menor_10'
        WHEN amount_abs < 50    THEN '10_50'
        WHEN amount_abs < 100   THEN '50_100'
        WHEN amount_abs < 500   THEN '100_500'
        WHEN amount_abs < 1000  THEN '500_1000'
        ELSE 'mayor_1000'
    END AS rango_monto,
    COUNT(*) AS total_transacciones,
    SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) AS total_fraudes,
    SUM(CASE WHEN is_fraud IN (0, 1) THEN 1 ELSE 0 END) AS transacciones_etiquetadas,
    ROUND(
        SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END)
        / SUM(CASE WHEN is_fraud IN (0, 1) THEN 1 ELSE 0 END) * 100,
        4
    ) AS tasa_fraude_pct
FROM silver.transactions_daniel
GROUP BY 1
ORDER BY
    CASE rango_monto
        WHEN 'menor_10' THEN 1
        WHEN '10_50' THEN 2
        WHEN '50_100' THEN 3
        WHEN '100_500' THEN 4
        WHEN '500_1000' THEN 5
        ELSE 6
    END;

## Nota sobre CASE WHEN

Inicialmente se intentaron usar rangos con símbolo `$`, por ejemplo `'$10 - $50'`, pero Databricks SQL puede interpretar `$` como parámetro dentro de notebooks SQL.

Para evitar ese comportamiento, se usaron etiquetas sin símbolo monetario:

- `menor_10`
- `10_50`
- `50_100`
- `100_500`
- `500_1000`
- `mayor_1000`

La lógica del análisis no cambia; solo se ajustó el nombre visible de los rangos.

## Resumen de hallazgos para PR

- El pico de fraude mensual ocurrió en **diciembre de 2015**, con una tasa de fraude de **0.5278%**.
- La categoría de comercio con más fraude usando `HAVING` para más de 1,000 transacciones y tasa mayor a 5% fue **Computers, Computer Peripheral Equipment**, con **10.8338%**.
- `HAVING` se usa cuando el filtro depende de una agregación, por ejemplo `COUNT(*)`, `SUM(...)` o una tasa calculada después del `GROUP BY`.
- El tipo de tarjeta con mayor tasa de fraude en Gold fue **Debit (Prepaid)**.
- La tabla `silver.transactions_daniel` contiene **13,305,915 filas** y **48 columnas**.